# One system, two jurisdictions

> **Demonstration only:** frozen synthetic data, not evidence about any real decision. Reasonsmith reports evidence and refusals; it does not certify compliance or provide legal advice.

The first table is the payoff: the same shipped credit system is checked against the US `ecoa` pack and the EU `gdpr` and `eu_ai_act` packs. A verdict can diverge because a pack asks for a different signal or because its applicability gate is different; the table keeps that reason beside the result.

In [ ]:
import html

from IPython.display import HTML, display

from reasonsmith.demo import deployed_credit_system
from reasonsmith.report import check_conformance
from reasonsmith.spec import load_pack


def show_table(rows, columns, title):
    head = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '—')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"<h3>{html.escape(title)}</h3><table><thead><tr>{head}</tr></thead>"
                      f"<tbody>{body}</tbody></table>"))

def difference_reason(requirement, result):
    outcome = result["outcome"]
    if outcome == "not_applicable":
        return f"Pack scope {requirement.scope!r} is not reached: no regulatory class was declared."
    if outcome == "unattainable":
        missing = ", ".join(result.get("signals_missing", []))
        return f"The duty requires {missing}; this system does not expose it."
    text = requirement.rationale if outcome == "satisfied" else result["evidence_summary"]
    return text.split(". ", 1)[0].rstrip(".") + "."

system = deployed_credit_system()
rows = []
for jurisdiction, pack_name in [("US", "ecoa"), ("EU", "gdpr"), ("EU", "eu_ai_act")]:
    pack = load_pack(pack_name)
    report = check_conformance(system, pack)
    for requirement, result in zip(pack.requirements, report.to_dict()["results"], strict=True):
        rows.append({"Duty": f"{requirement.id} ({requirement.article_clause})",
                     "Jurisdiction": jurisdiction, "Verdict": result["verdict"],
                     "Rung": result["strength"] or "—",
                     "Why this landed here": difference_reason(requirement, result)})
headline = ("ecoa_reg_b_1002_9_b_2_principal_reasons_complete",
            "gdpr_art22_3_safeguards_human_intervention",
            "gdpr_art22_1_automated_decision_prohibition",
            "eu_ai_act_art86_1_main_elements_of_the_decision")
rows.sort(key=lambda row: next((i for i, p in enumerate(headline) if row["Duty"].startswith(p)), 4))
show_table(rows, ["Duty", "Jurisdiction", "Verdict", "Rung", "Why this landed here"],
           "One frozen system across US and EU packs")


The divergence is substantive where the pack asks a different question: the ECOA and EU AI Act reason-completeness duties both use the exposed inference artefact, while GDPR's duties ask for signals this system does not emit. The EU AI Act rows are **not applicable**, not passes or failures: its shipped requirements are scoped to `high-risk`, and this system declares no regulatory class. That class is never inferred.

The legal wording and formalised limits come from the shipped pack files (`src/reasonsmith/packs/ecoa.toml`, `gdpr.toml`, and `eu_ai_act.toml`), whose source record is [`docs/legal-sources.md`](../docs/legal-sources.md). The applicability and evidence semantics are documented in [`docs/semantics.md`](../docs/semantics.md).